# Import libraries

In [70]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Get the data

In [ ]:
# Load the dataset from the CSV file
dataset = pd.read_csv('../data/book700k-800k.csv')
df = pd.DataFrame({'Id': dataset['Id'],
                        'Name': dataset['Name'],
                        'Authors': dataset['Authors'],
                        'Publish year': dataset['PublishYear'],
                        'Rating': dataset['Rating'],
                        'Description': dataset['Description']})
df['Text feature'] = (df['Authors'].fillna('') + ' ' +
                      df['Name'].fillna('') + ' ' +
                      df['Description'].fillna('') +
                      df['Description'].fillna(''))

# Display the first few rows of the dataset
df.head


<bound method NDFrame.head of            Id                                               Name  \
0      700000  A Passion to Preserve: Gay Men as Keepers of C...   
1      700002  Culture Keepers-Florida: Oral History of the A...   
2      700003  Holiday Favorites: The Best of the Williams-So...   
3      700004  Soups, Salads & Starters: the Best of Williams...   
4      700005                              Breakfasts & Brunches   
...       ...                                                ...   
54268  799991           Piano Concerto Highlights for Solo Piano   
54269  799993  Noggin King of the Nogs (The Sagas of Noggin t...   
54270  799994  No Greater Glory: The Four Immortal Chaplains ...   
54271  799996  The White Company by Arthur Conan Doyle, Ficti...   
54272  799997                    Livewire Real Lives Dawn Fraser   

                     Authors  Publish year  Rating  \
0               Will Fellows          2005    3.75   
1      Deborah Johnson-Simon          2006   

In [72]:
df['Text feature'][0]

'Will Fellows A Passion to Preserve: Gay Men as Keepers of Culture From large cities to rural communities, gay men have long been impassioned pioneers as keepers of culture: rescuing and restoring decrepit buildings, revitalizing blighted neighborhoods, saving artifacts and documents of historical significance. <i>A Passion to Preserve</i> explores this authentic and complex dimension of gay men’s lives by profiling early and contemporary preservationists from throughout the United States, highlighting contributions to the larger culture that gays are exceptionally inclined to make.'

In [73]:
# Replace Nan values with ''
df['Description'] = df['Description'].fillna('')
ori_description = df['Description']

In [74]:
df['Description'][0]

'From large cities to rural communities, gay men have long been impassioned pioneers as keepers of culture: rescuing and restoring decrepit buildings, revitalizing blighted neighborhoods, saving artifacts and documents of historical significance. <i>A Passion to Preserve</i> explores this authentic and complex dimension of gay men’s lives by profiling early and contemporary preservationists from throughout the United States, highlighting contributions to the larger culture that gays are exceptionally inclined to make.'

# Data preprocessing

In [75]:
import re
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

# Initialize the lemmatizer
lemmatizer = WordNetLemmatizer()

# Initialize the stopwords
stop_words = set(stopwords.words('english'))

def preprocessing_text(text):
    # Convert the input text to string
    text = str(text)
    
    # Convert text to lowercase
    text = text.lower()
    
    # Remove special characters and digits and replace them with a spcae
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    
    # Tokenize the text
    tokens = nltk.word_tokenize(text)
    
    # Remove stop words
    tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatize the tokens (convert words into their base dictionary form, ex. cats->cat)
    tokens = [lemmatizer.lemmatize(word, pos='v') for word in tokens]
    
    # Return the processed text as a string
    return " ".join(tokens)


def preprocess_dataframe(df, column_name):
    df[column_name]= df[column_name].apply(preprocessing_text)
    return df

df = preprocess_dataframe(df=df, column_name='Description')


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\thlam\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\thlam\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\thlam\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [76]:
df['Description'].head

<bound method NDFrame.head of 0        large cities rural communities gay men long im...
1                                                         
2        collector edition feature fabulous full color ...
3                                                         
4        america respect cookware retailer world larges...
                               ...                        
54268    concertos every pianist pinnacle performance r...
54269    king nogs br ice dragon br fly machine br omru...
54270    sink dorchester icy water greenland shortly mi...
54271    hilt cry cross narrow sea would find thick be ...
54272    book tell story dawn fraser vote greatest fema...
Name: Description, Length: 54273, dtype: str>

# Get text features using tf-idf

In [77]:
from sklearn.feature_extraction.text import TfidfVectorizer

def get_text_feature(df):
    text_features = df['Text feature']
    # Text feature
    tfidf = TfidfVectorizer(stop_words="english",
                            strip_accents='ascii',
                            token_pattern=r'\w+')

    tfidf_matrix = tfidf.fit_transform(text_features)
    tfidf.get_feature_names_out()
    
    return tfidf_matrix
tfidf_matrix = get_text_feature(df=df)

# Calculate cosine similarity
`from sklearn.metrics.pairwise import cosine_similarity`

In [78]:
from sklearn.metrics.pairwise import cosine_similarity

def calculate_cosine_similarity_of_a_target_book(book, tfidf_matrix):
    """Calculate the cosine similarity between a target book and all books
    in the TF-IDF matrix.

    Args:
        book (scipy.sparse.csr_matrix): 
            A single TF-IDF row vector representing the target book.
            Shape should be (1, n_features).
        tfidf_matrix (scipy.sparse.csr_matrix_): 
            TF-IDF matrix containing all book vectors.
            Shape should be (n_books, n_features).

    Returns:
        numpy.ndarray: A 2D array containing cosine similarity scores between the target book and 
        every book in the TF-IDF matrix.
        Shape will be (1, n_books).
    """
    X = book
    Y = tfidf_matrix
    cs = cosine_similarity(X, Y)
    return cs

book = calculate_cosine_similarity_of_a_target_book(tfidf_matrix=tfidf_matrix, book=tfidf_matrix[0])
book

array([[1.        , 0.1759361 , 0.        , ..., 0.01665048, 0.        ,
        0.01377254]], shape=(1, 54273))

# Add indices to the similarity array

In [79]:
tfidf_matrix.shape[0]

54273

In [80]:
def add_indices(sim_array):
    indexed_array = []
    for i, s in zip(range(tfidf_matrix.shape[0]), sim_array[0]):
        indexed_array.append((i, s))
    return indexed_array
    
book_idx = add_indices(book)
book_idx

[(0, np.float64(1.0000000000000004)),
 (1, np.float64(0.1759361010966507)),
 (2, np.float64(0.0)),
 (3, np.float64(0.0)),
 (4, np.float64(0.0)),
 (5, np.float64(0.004362876508365917)),
 (6, np.float64(0.0)),
 (7, np.float64(0.0)),
 (8, np.float64(0.02618616693317984)),
 (9, np.float64(0.003625613727191063)),
 (10, np.float64(0.004785675674866886)),
 (11, np.float64(0.002029077582183373)),
 (12, np.float64(0.027821324639495902)),
 (13, np.float64(0.031798158817712134)),
 (14, np.float64(0.019411515560834834)),
 (15, np.float64(0.0)),
 (16, np.float64(0.0)),
 (17, np.float64(0.0)),
 (18, np.float64(0.0025114041824299653)),
 (19, np.float64(0.013737418330807183)),
 (20, np.float64(0.0)),
 (21, np.float64(0.0)),
 (22, np.float64(0.029205883127773797)),
 (23, np.float64(0.0)),
 (24, np.float64(0.025214664716739867)),
 (25, np.float64(0.013690511806384871)),
 (26, np.float64(0.0)),
 (27, np.float64(0.0)),
 (28, np.float64(0.0)),
 (29, np.float64(0.0)),
 (30, np.float64(0.0)),
 (31, np.float6

# Sort descendent

In [81]:
book_idx

[(0, np.float64(1.0000000000000004)),
 (1, np.float64(0.1759361010966507)),
 (2, np.float64(0.0)),
 (3, np.float64(0.0)),
 (4, np.float64(0.0)),
 (5, np.float64(0.004362876508365917)),
 (6, np.float64(0.0)),
 (7, np.float64(0.0)),
 (8, np.float64(0.02618616693317984)),
 (9, np.float64(0.003625613727191063)),
 (10, np.float64(0.004785675674866886)),
 (11, np.float64(0.002029077582183373)),
 (12, np.float64(0.027821324639495902)),
 (13, np.float64(0.031798158817712134)),
 (14, np.float64(0.019411515560834834)),
 (15, np.float64(0.0)),
 (16, np.float64(0.0)),
 (17, np.float64(0.0)),
 (18, np.float64(0.0025114041824299653)),
 (19, np.float64(0.013737418330807183)),
 (20, np.float64(0.0)),
 (21, np.float64(0.0)),
 (22, np.float64(0.029205883127773797)),
 (23, np.float64(0.0)),
 (24, np.float64(0.025214664716739867)),
 (25, np.float64(0.013690511806384871)),
 (26, np.float64(0.0)),
 (27, np.float64(0.0)),
 (28, np.float64(0.0)),
 (29, np.float64(0.0)),
 (30, np.float64(0.0)),
 (31, np.float6

In [82]:
def sort_des(idx_array):
    sorted_array = sorted(idx_array, key=lambda x: x[1], reverse=True)
    return sorted_array[1:11]

sorted_array = sort_des(idx_array=book_idx)
sorted_array

[(15012, np.float64(0.26822860898560036)),
 (13832, np.float64(0.24321452699078033)),
 (40495, np.float64(0.24237787245890857)),
 (23708, np.float64(0.23429471340904848)),
 (17247, np.float64(0.22106917446108396)),
 (16360, np.float64(0.21985578691770066)),
 (6879, np.float64(0.21816938715351064)),
 (5935, np.float64(0.2136734315443305)),
 (7916, np.float64(0.21320044389731063)),
 (39114, np.float64(0.201853633810062))]

# Get the book names from the indices

In [83]:
sorted_array[1:6]

[(13832, np.float64(0.24321452699078033)),
 (40495, np.float64(0.24237787245890857)),
 (23708, np.float64(0.23429471340904848)),
 (17247, np.float64(0.22106917446108396)),
 (16360, np.float64(0.21985578691770066))]

In [84]:
def get_book_name(rec_books):
    rec_books_info = []
    for i in rec_books:
        
        rec_books_info.append(df.iloc[i[0], :])
        
    return pd.DataFrame(rec_books_info)

result = get_book_name(sorted_array)
# result = pd.DataFrame(result)
result

,Id,Name,Authors,Publish year,Rating,Description,Text feature
15012,727604,The Soul Beneath the Skin: The Unseen Hearts a...,David Nimmons,2002,3.88,surprise think provoke book begin obvious fact...,David Nimmons The Soul Beneath the Skin: The U...
13832,725500,Life Outside: The Signorile Report on Gay Men:...,Michelangelo Signorile,1998,3.68,strong popular em em magazine columnist michel...,Michelangelo Signorile Life Outside: The Signo...
40495,774575,Gay Men at the Millennium,Michael Lowenthal,1997,4.40,core issue face gay community close millennium...,Michael Lowenthal Gay Men at the Millennium Co...
23708,743801,Lavender Culture,Karla Jay,1994,3.55,influence gays lesbians language literature th...,Karla Jay Lavender Culture The influence of ga...
17247,731667,Queer Wars: The New Gay Right and Its Critics,Paul A. Robinson,2006,4.05,rebellion stonewall recent battle sex marriage...,Paul A. Robinson Queer Wars: The New Gay Right...
16360,730029,"If You Seduce a Straight Person, Can You Make ...",John P. De Cecco,1993,4.00,debate whether people bear homosexual biologic...,John P. De Cecco If You Seduce a Straight Pers...
6879,712601,Art and Sex in Greenwich Village: A Memoir of ...,Felice Picano,2007,3.82,decade stonewall rebellions small gay press na...,Felice Picano Art and Sex in Greenwich Village...
5935,710934,Gay by the Bay: A History of Queer Culture in ...,Susan Stryker,1996,3.89,fabulous montage word image first book ever ch...,Susan Stryker Gay by the Bay: A History of Que...
7916,714426,Men Like Us : The GMHC Complete Guide to Gay M...,Daniel Wolfe,2000,4.06,offer practical advice gay men exercise diet m...,Daniel Wolfe Men Like Us : The GMHC Complete G...
39114,772020,John Gay and the London Theatre,Calhoun Winton,1993,3.00,beggar opera often refer today first musical c...,Calhoun Winton John Gay and the London Theatre...


# Put everything together

In [89]:
def get_rec_books(book_idx, df):
    selected_book = df.iloc[book_idx:book_idx+1, :]
    data = preprocess_dataframe(df=df, column_name='Text feature')
    matrix = get_text_feature(data)
    book_sim = calculate_cosine_similarity_of_a_target_book(book=matrix[book_idx], tfidf_matrix=matrix)
    book_sim = add_indices(book_sim)
    book_sim = sort_des(book_sim)
    result = get_book_name(rec_books=book_sim)
    
    return selected_book, result

selected_book, rec_books = get_rec_books(df=df, book_idx=1000)
    

In [90]:
rec_books

,Id,Name,Authors,Publish year,Rating,Description,Text feature
35161,764815,Beginning Ruby: From Novice to Professional,Peter Cooper,2007,3.83,ruby perhaps best know engine power hugely pop...,peter cooper begin ruby novice professional ru...
3610,706694,"This Way, Ruby!",Jonathan Emmett,2007,3.81,ruby brothers sisters always race search adven...,jonathan emmett way ruby ruby brothers sisters...
42804,778985,A Revolution in Eating: How the Quest for Food...,James McWilliams,2005,3.92,,jam mcwilliams revolution eat quest food shape...
3612,706696,The Ruby Way,Hal Fulton,2001,3.90,ruby way assume reader already familiar subjec...,hal fulton ruby way ruby way assume reader alr...
45891,784687,Bunny Party,Rosemary Wells,2003,3.72,grandma birthday max ruby party ruby invite se...,rosemary well bunny party grandma birthday max...
25912,747787,Country Loving,Julie Highmore,2003,3.11,life slow alarmingly moment ruby grant oliver ...,julie highmore country love life slow alarming...
37429,768954,Cow Crimes and the Mustang Menace (Ruby Taylor...,Sharon Dunn,2005,3.90,award win series continue mystery ruby taylor ...,sharon dunn cow crimes mustang menace ruby tay...
35794,765883,Ruby in a Nutshell,Yukihiro Matsumoto,2007,3.39,write yukihiro matsumoto matz creator language...,yukihiro matsumoto ruby nutshell write yukihir...
12496,723071,One Summer Night,Gerri Hill,2004,3.82,college professor johanna marshall swear would...,gerri hill one summer night college professor ...
45894,784690,Ruby's Beauty Shop,Rosemary Wells,2004,3.80,welcome ruby beauty shop guess max say max sis...,rosemary well ruby beauty shop welcome ruby be...


In [91]:
selected_book = pd.DataFrame(selected_book)
selected_book

,Id,Name,Authors,Publish year,Rating,Description,Text feature
1000,701907,"Diary of a Slave Girl, Ruby Jo",K.J. McWilliams,2001,4.26,,k j mcwilliams diary slave girl ruby jo


In [92]:
print(f"Selected book: '{selected_book['Name'].iloc[0]}'")
print(f"Selected book description: '{selected_book['Description'].iloc[0]}'\n")
for name in rec_books['Name']: 
    print(f"{name}")

Selected book: 'Diary of a Slave Girl, Ruby Jo'
Selected book description: ''

Beginning Ruby: From Novice to Professional
This Way, Ruby!
A Revolution in Eating: How the Quest for Food Shaped America
The Ruby Way
Bunny Party
Country Loving
Cow Crimes and the Mustang Menace (Ruby Taylor Mystery #3)
Ruby in a Nutshell
One Summer Night
Ruby's Beauty Shop
